In [ ]:
# Kaggle训练脚本
# 特点：
# 1. 代码和运行环境放在/tmp目录（不会被保存）
# 2. artifacts（模型、数据集）放在当前工作目录（会被保存）

import os
from pathlib import Path
from dotenv import load_dotenv

# 1. 加载环境变量
env_path = Path('/kaggle/input/env-key/.env')

if env_path.exists():
    load_dotenv(dotenv_path=env_path, override=True)
    print("✅ 环境变量已从数据集成功加载")
else:
    print("❌ 未找到 .env 文件，请检查数据集挂载路径")

# 验证加载结果
MODELSCOPE_TOKEN = os.getenv("MODELSCOPE_TOKEN")

!echo $MODELSCOPE_TOKEN
!nvidia-smi

# 2. 设置目录结构
# 代码和环境放在/tmp（不保存）
WORK_DIR = Path("/tmp/train-llm")
# artifacts放在当前目录（保存）
ARTIFACTS_DIR = Path("/kaggle/working/artifacts")

# 设置环境变量，让所有脚本都使用这个artifacts目录
os.environ['ARTIFACTS_DIR'] = str(ARTIFACTS_DIR)

print(f"📁 工作目录（临时）: {WORK_DIR}")
print(f"📁 Artifacts目录（持久化）: {ARTIFACTS_DIR}")

# 3. 克隆代码到临时目录
!cd /tmp && git clone https://github.com/try-agaaain/train-llm.git

# 4. 安装依赖到临时目录
!pip install uv
!cd {WORK_DIR} && uv sync --index-url https://pypi.org/simple

# 5. 运行评估（artifacts会保存到/kaggle/working/artifacts）
!cd {WORK_DIR} && make dpull MODELSCOPE_TOKEN=$MODELSCOPE_TOKEN ARTIFACTS_DIR={ARTIFACTS_DIR}
!cd {WORK_DIR} && make mpull MODELSCOPE_TOKEN=$MODELSCOPE_TOKEN ARTIFACTS_DIR={ARTIFACTS_DIR}
!cd {WORK_DIR} && make evaluate EVAL_BATCH_SIZE=8 ARTIFACTS_DIR={ARTIFACTS_DIR}

# 6. 推送数据集到ModelScope
!cd {WORK_DIR} && MODELSCOPE_TOKEN=$MODELSCOPE_TOKEN make dpush ARTIFACTS_DIR={ARTIFACTS_DIR}

print("\n✅ 训练完成！")
print(f"📦 Artifacts已保存在: {ARTIFACTS_DIR}")
